# Forecasting Patent Innovation at Ericsson
BMAN60422 group coursework: annual patent forecasting (Task 2) and text-based segmentation forecasting (Task 3).

Place `ericsson_patent_rich_dataset.csv` (not included in this repo) in the same folder, then **Run All**.

## Task 2: Baseline forecasting models
Decision Tree (max_depth=2) and Random Forest (n_estimators=35), using lag-1 count plus 6 lagged keyword proportions, with a chronological 80/10/10 split.

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_percentage_error,
                             mean_absolute_error,
                             r2_score)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_PATH        = "ericsson_patent_rich_dataset.csv"
RANDOM_STATE     = 123
KEYWORDS         = ["kw_iot", "kw_ai_ml", "kw_security",
                    "kw_energy", "kw_network", "kw_data"]

DT_DEPTH_GRID    = [2, 4, 6]
RF_NEST_GRID     = list(range(10, 51, 5))    # 10, 15, 20, ..., 50

DT_BEST_DEPTH    = 2
RF_BEST_NEST     = 35


# ---------------------------------------------------------------------------
# 1. Load and aggregate to yearly resolution
# ---------------------------------------------------------------------------
def load_and_aggregate(path: str) -> pd.DataFrame:
    """Load raw patent rows and aggregate to yearly counts + keyword shares.

    Excludes 2025 (partial year).  Keyword 'proportion' for a year is
    computed as keyword_count / total_patents in that year — which equals
    the mean of the binary kw_* flag.
    """
    df = pd.read_csv(path)
    df = df[df["year"] != 2025].copy()

    yearly = (df.groupby("year")
                .agg(total=("patent_id", "count"),
                     **{k: (k, "mean") for k in KEYWORDS})
                .reset_index()
                .sort_values("year")
                .reset_index(drop=True))
    return yearly


# ---------------------------------------------------------------------------
# 2. Build lag-1 features
# ---------------------------------------------------------------------------
def build_lag_features(yearly: pd.DataFrame):
    """Create lag_1 of the target and lag_1 of each keyword proportion.

    Drops the first row (1976), which has no prior year.
    """
    df = yearly.copy()
    df["lag_1"] = df["total"].shift(1)
    for k in KEYWORDS:
        df[f"lag_{k}"] = df[k].shift(1)

    feature_cols = ["lag_1"] + [f"lag_{k}" for k in KEYWORDS]
    df = df.dropna(subset=feature_cols).reset_index(drop=True)
    return df, feature_cols


# ---------------------------------------------------------------------------
# 3. Chronological 80/10/10 split
# ---------------------------------------------------------------------------
def chronological_split(df: pd.DataFrame):
    """80/10/10 split preserving temporal order."""
    n       = len(df)
    n_train = int(round(n * 0.8))
    n_val   = int(round(n * 0.1))

    train = df.iloc[:n_train].reset_index(drop=True)
    val   = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test  = df.iloc[n_train + n_val:].reset_index(drop=True)
    return train, val, test


# ---------------------------------------------------------------------------
# 4. Metric helpers
# ---------------------------------------------------------------------------
def mape(y, pred):
    return mean_absolute_percentage_error(y, pred) * 100


def evaluate(model, X, y) -> dict:
    """Return MAPE, MAE, R² for a fitted model on (X, y)."""
    pred = model.predict(X)
    return {
        "MAPE": mape(y, pred),
        "MAE":  mean_absolute_error(y, pred),
        "R2":   r2_score(y, pred),
    }


# ---------------------------------------------------------------------------
# 5. Hyperparameter sweeps
# ---------------------------------------------------------------------------
def sweep_dt(X_train, y_train, X_val, y_val, X_test, y_test, depth_grid):
    """Decision-Tree sweep over max_depth.  Returns a tidy DataFrame."""
    rows = []
    for d in depth_grid:
        mdl = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
        tr, va, te = evaluate(mdl, X_train, y_train), evaluate(mdl, X_val, y_val), evaluate(mdl, X_test, y_test)
        rows.append({
            "max_depth":   d,
            "Train MAPE":  f"{tr['MAPE']:.2f}%",
            "Val MAPE":    f"{va['MAPE']:.2f}%",
            "Test MAPE":   f"{te['MAPE']:.2f}%",
            "Train R²":    f"{tr['R2']:.3f}",
            "Val R²":      f"{va['R2']:.3f}",
            "Test R²":     f"{te['R2']:.3f}",
        })
    return pd.DataFrame(rows)


def sweep_rf(X_train, y_train, X_val, y_val, X_test, y_test, nest_grid):
    """Random-Forest sweep over n_estimators.  Returns a tidy DataFrame."""
    rows = []
    for n_est in nest_grid:
        mdl = RandomForestRegressor(n_estimators=n_est, random_state=RANDOM_STATE).fit(X_train, y_train)
        tr, va, te = evaluate(mdl, X_train, y_train), evaluate(mdl, X_val, y_val), evaluate(mdl, X_test, y_test)
        rows.append({
            "n_estimators": n_est,
            "Train MAPE":   f"{tr['MAPE']:.2f}%",
            "Val MAPE":     f"{va['MAPE']:.2f}%",
            "Test MAPE":    f"{te['MAPE']:.2f}%",
            "Train R²":     f"{tr['R2']:.3f}",
            "Val R²":       f"{va['R2']:.3f}",
            "Test R²":      f"{te['R2']:.3f}",
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 6. Persistence baseline (diagnostic)
# ---------------------------------------------------------------------------
def persistence_predictions(subset: pd.DataFrame):
    """Persistence: y_hat_t = y_{t-1} (i.e. just the lag_1 column)."""
    return subset["lag_1"].values


# ---------------------------------------------------------------------------
# 7. Main pipeline
# ---------------------------------------------------------------------------
def main():
    # ---- Data prep ----
    yearly         = load_and_aggregate(DATA_PATH)
    data, feats    = build_lag_features(yearly)
    train, val, test = chronological_split(data)

    print(f"Total years (after lagging): {len(data)} "
          f"({data['year'].min()}–{data['year'].max()})")
    print(f"  Train: {train['year'].min()}–{train['year'].max()}  (n={len(train)})")
    print(f"  Val:   {val['year'].min()}–{val['year'].max()}    (n={len(val)})")
    print(f"  Test:  {test['year'].min()}–{test['year'].max()}    (n={len(test)})\n")

    X_train, y_train = train[feats], train["total"]
    X_val,   y_val   = val[feats],   val["total"]
    X_test,  y_test  = test[feats],  test["total"]

    # ====================================================================
    #  Hyperparameter sweeps
    # ====================================================================
    print("══════════ Decision Tree sweep — max_depth ∈ {2, 4, 6} ══════════")
    dt_sweep = sweep_dt(X_train, y_train, X_val, y_val, X_test, y_test, DT_DEPTH_GRID)
    print(dt_sweep.to_string(index=False))
    print(f"\n→ Selected: max_depth = {DT_BEST_DEPTH} "
          f"(parsimonious, robust on test, no overfitting)\n")

    print("══════════ Random Forest sweep — n_estimators ∈ {10, 15, …, 50} ══════════")
    rf_sweep = sweep_rf(X_train, y_train, X_val, y_val, X_test, y_test, RF_NEST_GRID)
    print(rf_sweep.to_string(index=False))
    print(f"\n→ Selected: n_estimators = {RF_BEST_NEST} "
          f"(sits on the stable plateau, best test MAPE / R² in the lower range)\n")

    # ====================================================================
    #  Fit locked champion models on TRAIN
    # ====================================================================
    dt = DecisionTreeRegressor(max_depth=DT_BEST_DEPTH,
                               random_state=RANDOM_STATE).fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=RF_BEST_NEST,
                               random_state=RANDOM_STATE).fit(X_train, y_train)

    for label, mdl in [(f"Decision Tree (max_depth={DT_BEST_DEPTH})", dt),
                       (f"Random Forest (n_estimators={RF_BEST_NEST})", rf)]:
        tr, va, te = evaluate(mdl, X_train, y_train), evaluate(mdl, X_val, y_val), evaluate(mdl, X_test, y_test)
        out = pd.DataFrame(
            [[tr['MAPE'], tr['MAE'], tr['R2']],
             [va['MAPE'], va['MAE'], va['R2']],
             [te['MAPE'], te['MAE'], te['R2']]],
            index=["Train", "Val", "Test"],
            columns=["MAPE", "MAE", "R²"]).round(3)
        print(f"── {label} ──")
        print(out.to_string())
        print()

    # ---- Persistence baseline (diagnostic) ----
    print("── Persistence baseline (y_hat_t = y_{t-1}) ──")
    pv, pt = persistence_predictions(val), persistence_predictions(test)
    print(f"  Val  MAPE: {mape(y_val, pv):.2f}%   R²: {r2_score(y_val, pv):.3f}")
    print(f"  Test MAPE: {mape(y_test, pt):.2f}%   R²: {r2_score(y_test, pt):.3f}\n")

    # ---- MDI feature importance ----
    print("── MDI feature importance ──")
    mdi = pd.DataFrame({
        f"DT (max_depth={DT_BEST_DEPTH})":   pd.Series(dt.feature_importances_, index=feats),
        f"RF (n_est={RF_BEST_NEST})":        pd.Series(rf.feature_importances_, index=feats),
    }).round(4)
    print(mdi.to_string())
    print()

    # ---- 2025 forward forecast (refit on full 1977-2024) ----
    dt_full = DecisionTreeRegressor(max_depth=DT_BEST_DEPTH,
                                    random_state=RANDOM_STATE).fit(data[feats], data["total"])
    rf_full = RandomForestRegressor(n_estimators=RF_BEST_NEST,
                                    random_state=RANDOM_STATE).fit(data[feats], data["total"])

    row_2024 = yearly[yearly["year"] == 2024].iloc[0]
    x_2025 = pd.DataFrame([{
        "lag_1": row_2024["total"],
        **{f"lag_{k}": row_2024[k] for k in KEYWORDS}
    }])[feats]

    print("── 2025 forecast (refit on full 1977-2024) ──")
    print(f"  DT (max_depth={DT_BEST_DEPTH}):     {float(dt_full.predict(x_2025)[0]):.0f}")
    print(f"  RF (n_estimators={RF_BEST_NEST}):  {float(rf_full.predict(x_2025)[0]):.0f}")


main()

Total years (after lagging): 48 (1977–2024)
  Train: 1977–2014  (n=38)
  Val:   2015–2019    (n=5)
  Test:  2020–2024    (n=5)

══════════ Decision Tree sweep — max_depth ∈ {2, 4, 6} ══════════
 max_depth Train MAPE Val MAPE Test MAPE Train R² Val R² Test R²
         2     64.50%   11.26%     5.53%    0.911 -2.471  -0.005
         4     22.19%   11.89%    22.52%    0.995 -3.426  -7.740
         6      7.68%    8.52%    10.23%    1.000 -1.238  -1.118

→ Selected: max_depth = 2 (parsimonious, robust on test, no overfitting)

══════════ Random Forest sweep — n_estimators ∈ {10, 15, …, 50} ══════════
 n_estimators Train MAPE Val MAPE Test MAPE Train R² Val R² Test R²
           10     16.80%   20.68%    10.07%    0.945 -9.830  -1.241
           15     15.88%   20.72%    10.12%    0.948 -9.536  -1.101
           20     18.14%   18.13%     8.81%    0.961 -7.283  -0.615
           25     17.28%   16.27%     7.13%    0.964 -5.930  -0.345
           30     16.41%   16.08%     6.83%    0.970 -6.

## Task 3: Text-embedding cluster forecasting
TF-IDF (scikit-learn English and domain stopwords) → Truncated SVD (50) → L2 normalisation → K-Means (K=3), followed by a per-cluster DT/RF forecast with lagged features, summed across clusters.

### Setup

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor


# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION (locked values from K-selection analysis)
# ──────────────────────────────────────────────────────────────────────────────
DATA_PATH        = 'ericsson_patent_rich_dataset.csv'
YEAR_MAX         = 2024            # exclude partial-year 2025 from analysis
RANDOM_STATE     = 123
N_SVD_COMPONENTS = 50               # locked (report: 50 more stable than 100)
K_CLUSTERS       = 3                # locked K (report Section 4.1)
N_INIT_KMEANS    = 10
SPLIT_TRAIN_FRAC = 0.80             # chronological 80/10/10 split (matches Task 2)
SPLIT_VAL_FRAC   = 0.10
RF_N_ESTIMATORS  = 35               # locked from Task 2 baseline
DT_MAX_DEPTH     = 2                # locked from Task 2 baseline

# Stopwords for TF-IDF
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOPWORDS_BASIC = list(ENGLISH_STOP_WORDS)   # scikit-learn English stopword list
STOPWORDS_DOMAIN = ['method','system','communication','apparatus','device']

# Keyword features (same 6 used in Task 2 locked baseline)
KW_FOR_FEATURES = ['kw_ai_ml','kw_security','kw_iot','kw_network','kw_energy','kw_data']
FEATS = ['lag_1'] + [f'prop_{k}' for k in KW_FOR_FEATURES]


# ──────────────────────────────────────────────────────────────────────────────
# UTILITY FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────

def safe_mape(y_true, y_pred):
    """MAPE excluding zero-actual years (division by zero is undefined)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def build_singularisation_map(titles, stopwords, min_df=5):
    """Auto-detect plural→singular pairs in vocabulary using rule-based suffix
    matching: 's', 'es', 'ies' endings. Each pair requires both forms to
    appear in the corpus at min_df=5."""
    cv = CountVectorizer(ngram_range=(1, 1), min_df=min_df,
                         stop_words=stopwords, lowercase=False, binary=True)
    cv.fit(titles)
    vocab = set(cv.get_feature_names_out())

    mapping = {}
    for w in cv.get_feature_names_out():
        if len(w) < 4:
            continue
        # Skip words that look plural but aren't (e.g., "wireless", "apparatus", "basis")
        if w.endswith('ss') or w.endswith('us') or w.endswith('is'):
            continue
        candidate_singular = None
        # Rule 1: 'ies' ending → 'y' (frequencies → frequency)
        if w.endswith('ies') and len(w) > 4:
            sing = w[:-3] + 'y'
            if sing in vocab:
                candidate_singular = sing
        # Rule 2: 'es' ending → drop 'es' (processes → process); fallback to drop 's'
        elif w.endswith('es') and len(w) > 4:
            sing = w[:-2]
            if sing in vocab:
                candidate_singular = sing
            else:
                sing_s = w[:-1]
                if sing_s in vocab:
                    candidate_singular = sing_s
        # Rule 3: 's' ending → drop 's'
        elif w.endswith('s'):
            sing = w[:-1]
            if sing in vocab:
                candidate_singular = sing
        if candidate_singular:
            mapping[w] = candidate_singular
    return mapping


def apply_singularisation(text, mapping):
    """Replace each plural token in text with its singular form per the mapping."""
    return ' '.join(mapping.get(t, t) for t in text.split())


def build_cluster_panel(cluster_df, year_range, kw_cols):
    """For one cluster's patents, build the annual panel of patent_count and
    prop_kw_* features. Years with zero patents get count=0 and prop_kw=0."""
    yearly = cluster_df.groupby('year').agg(
        patent_count=('patent_id', 'size'),
        **{f'prop_{k}': (k, 'mean') for k in kw_cols}
    )
    yearly = yearly.reindex(year_range, fill_value=0)
    for col in [f'prop_{k}' for k in kw_cols]:
        yearly[col] = yearly[col].fillna(0)
    return yearly.reset_index().rename(columns={'index': 'year'})

### STEP 1 — LOAD DATA

In [3]:
print("=" * 80)
print("STEP 1 — Loading and filtering dataset")
print("=" * 80)

df = pd.read_csv(DATA_PATH)
df = df[df['year'] <= YEAR_MAX].reset_index(drop=True)
print(f"Loaded {len(df)} patents (years {df['year'].min()}-{df['year'].max()})")

STEP 1 — Loading and filtering dataset
Loaded 28941 patents (years 1976-2024)


### STEP 2 — TEXT PREPROCESSING

In [4]:
print("\n" + "=" * 80)
print("STEP 2 — Text preprocessing (lowercase + strip punctuation)")
print("=" * 80)

titles_clean = (df['patent_title'].astype(str)
                  .str.lower()
                  .str.replace(r'[^\w\s]', ' ', regex=True)
                  .str.replace(r'\s+', ' ', regex=True)
                  .str.strip())
print(f"Cleaned {len(titles_clean)} titles")


STEP 2 — Text preprocessing (lowercase + strip punctuation)
Cleaned 28941 titles


### STEP 3 — SINGULARISATION (plural→singular pairs)

In [5]:
print("\n" + "=" * 80)
print("STEP 3 — Build and apply plural→singular mapping")
print("=" * 80)

singular_map = build_singularisation_map(titles_clean, STOPWORDS_BASIC)
print(f"Detected {len(singular_map)} plural→singular pairs")

titles_processed = titles_clean.apply(lambda t: apply_singularisation(t, singular_map))


STEP 3 — Build and apply plural→singular mapping
Detected 316 plural→singular pairs


### STEP 4 — TF-IDF VECTORISATION

In [6]:
print("\n" + "=" * 80)
print("STEP 4 — TF-IDF vectorisation (ngrams 1-2, min_df=5, max_df=0.5, L2 norm)")
print("=" * 80)

stopwords_full = STOPWORDS_BASIC + STOPWORDS_DOMAIN
vectoriser = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.5,
    sublinear_tf=True,
    norm='l2',
    stop_words=stopwords_full,
    lowercase=False,
)
X_tfidf = vectoriser.fit_transform(titles_processed)
print(f"TF-IDF matrix: {X_tfidf.shape[0]:,} docs × {X_tfidf.shape[1]:,} terms")


STEP 4 — TF-IDF vectorisation (ngrams 1-2, min_df=5, max_df=0.5, L2 norm)
TF-IDF matrix: 28,941 docs × 6,412 terms


### STEP 5 — TRUNCATED SVD (n=50)

In [7]:
print("\n" + "=" * 80)
print(f"STEP 5 — Truncated SVD (n_components={N_SVD_COMPONENTS})")
print("=" * 80)

svd = TruncatedSVD(
    n_components=N_SVD_COMPONENTS,
    random_state=RANDOM_STATE,
    algorithm='randomized',
)
X_svd = svd.fit_transform(X_tfidf)
print(f"SVD output: {X_svd.shape}, cumulative variance: {svd.explained_variance_ratio_.sum():.4f}")


STEP 5 — Truncated SVD (n_components=50)
SVD output: (28941, 50), cumulative variance: 0.1533


### STEP 6 — L2-NORMALISE SVD OUTPUT

In [8]:
print("\n" + "=" * 80)
print("STEP 6 — L2-normalise SVD output")
print("=" * 80)

X_emb = normalize(X_svd, norm='l2')
print(f"Embedding shape: {X_emb.shape}, sample row norm: {np.linalg.norm(X_emb[0]):.4f}")


STEP 6 — L2-normalise SVD output
Embedding shape: (28941, 50), sample row norm: 1.0000


### STEP 7 — K-MEANS CLUSTERING (K=3)

In [9]:
print("\n" + "=" * 80)
print(f"STEP 7 — K-Means clustering (K={K_CLUSTERS})")
print("=" * 80)

kmeans = KMeans(
    n_clusters=K_CLUSTERS,
    n_init=N_INIT_KMEANS,
    init='k-means++',
    random_state=42,
)
cluster_labels = kmeans.fit_predict(X_emb)

df['cluster'] = cluster_labels
cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
print(f"\nCluster sizes:")
for c, n in cluster_sizes.items():
    pct = n / len(df) * 100
    print(f"  Cluster {c}: {n:>6} patents ({pct:>5.1f}%)")


STEP 7 — K-Means clustering (K=3)

Cluster sizes:
  Cluster 0:   5432 patents ( 18.8%)
  Cluster 1:   5871 patents ( 20.3%)
  Cluster 2:  17638 patents ( 60.9%)


### Top terms per cluster

In [10]:
terms = vectoriser.get_feature_names_out()
centroids = svd.inverse_transform(kmeans.cluster_centers_)
for c in range(K_CLUSTERS):
    top = centroids[c].argsort()[::-1][:5]
    print(f"Cluster {c}: {', '.join(terms[j] for j in top)}")

Cluster 0: network, node, wireless network, network node, wireless
Cluster 1: using, based, digital, detection, providing
Cluster 2: arrangement, control, radio, signal, data


### STEP 8 — BUILD PER-CLUSTER ANNUAL PANELS WITH LAG_1 AND PROP_KW FEATURES

In [11]:
print("\n" + "=" * 80)
print("STEP 8 — Build per-cluster annual panels (lag_1 + 6 lagged prop_kw features)")
print("=" * 80)

year_range = range(int(df['year'].min()), int(df['year'].max()) + 1)
cluster_panels = {}
for c in range(K_CLUSTERS):
    panel = build_cluster_panel(df[df['cluster'] == c], year_range, KW_FOR_FEATURES)
    panel['lag_1'] = panel['patent_count'].shift(1)
    for k in KW_FOR_FEATURES:                       # previous year's keyword share (no look-ahead)
        panel[f'prop_{k}'] = panel[f'prop_{k}'].shift(1)
    panel = panel.dropna(subset=['lag_1']).reset_index(drop=True)
    cluster_panels[c] = panel

n_years_total = len(cluster_panels[0])
n_train = int(round(n_years_total * SPLIT_TRAIN_FRAC))
n_val   = int(round(n_years_total * SPLIT_VAL_FRAC))

print(f"Total usable years per cluster: {n_years_total}")
print(f"Chronological split: train={n_train}, val={n_val}, test={n_years_total - n_train - n_val}")
print(f"  Train: {cluster_panels[0]['year'].iloc[0]}-{cluster_panels[0]['year'].iloc[n_train-1]}")
print(f"  Val:   {cluster_panels[0]['year'].iloc[n_train]}-{cluster_panels[0]['year'].iloc[n_train+n_val-1]}")
print(f"  Test:  {cluster_panels[0]['year'].iloc[n_train+n_val]}-{cluster_panels[0]['year'].iloc[-1]}")


STEP 8 — Build per-cluster annual panels (lag_1 + 6 lagged prop_kw features)
Total usable years per cluster: 48
Chronological split: train=38, val=5, test=5
  Train: 1977-2014
  Val:   2015-2019
  Test:  2020-2024


### STEP 9 — TRAIN RF AND DT PER CLUSTER

In [12]:
print("\n" + "=" * 80)
print(f"STEP 9 — Train RF n={RF_N_ESTIMATORS} and DT depth={DT_MAX_DEPTH} per cluster")
print("=" * 80)

per_cluster_predictions = {}

for c in range(K_CLUSTERS):
    p = cluster_panels[c]
    train = p.iloc[:n_train]
    val   = p.iloc[n_train:n_train + n_val]
    test  = p.iloc[n_train + n_val:]

    rf_model = RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        random_state=RANDOM_STATE,
    ).fit(train[FEATS], train['patent_count'])
    dt_model = DecisionTreeRegressor(
        max_depth=DT_MAX_DEPTH,
        random_state=RANDOM_STATE,
    ).fit(train[FEATS], train['patent_count'])

    per_cluster_predictions[c] = {
        'years_tr':   train['year'].values,
        'years_va':   val['year'].values,
        'years_te':   test['year'].values,
        'actual_tr':  train['patent_count'].values,
        'actual_va':  val['patent_count'].values,
        'actual_te':  test['patent_count'].values,
        'rf_pred_tr': rf_model.predict(train[FEATS]),
        'rf_pred_va': rf_model.predict(val[FEATS]),
        'rf_pred_te': rf_model.predict(test[FEATS]),
        'dt_pred_tr': dt_model.predict(train[FEATS]),
        'dt_pred_va': dt_model.predict(val[FEATS]),
        'dt_pred_te': dt_model.predict(test[FEATS]),
    }

print(f"Trained {K_CLUSTERS} RF models and {K_CLUSTERS} DT models (one per cluster)")


STEP 9 — Train RF n=35 and DT depth=2 per cluster
Trained 3 RF models and 3 DT models (one per cluster)


### STEP 10 — AGGREGATE CLUSTER FORECASTS AND COMPUTE MAPE

In [13]:
print("\n" + "=" * 80)
print("STEP 10 — Aggregate cluster forecasts → aggregate train/val/test MAPE")
print("=" * 80)

def aggregate_split(split_suffix):
    """Sum actuals and predictions across all clusters for one split."""
    n_yrs = len(per_cluster_predictions[0][f'years_{split_suffix}'])
    actual_sum = np.zeros(n_yrs)
    rf_sum     = np.zeros(n_yrs)
    dt_sum     = np.zeros(n_yrs)
    for c in range(K_CLUSTERS):
        d = per_cluster_predictions[c]
        actual_sum += d[f'actual_{split_suffix}']
        rf_sum     += d[f'rf_pred_{split_suffix}']
        dt_sum     += d[f'dt_pred_{split_suffix}']
    years = per_cluster_predictions[0][f'years_{split_suffix}']
    return years, actual_sum, rf_sum, dt_sum

print(f"\n{'Split':<10}{'Years':<14}{'#yrs':>6}{'RF MAPE':>12}{'DT MAPE':>12}")
print("-" * 56)
for split, label in [('tr', 'Train'), ('va', 'Val'), ('te', 'Test')]:
    years, actual_sum, rf_sum, dt_sum = aggregate_split(split)
    rf_mape = safe_mape(actual_sum, rf_sum)
    dt_mape = safe_mape(actual_sum, dt_sum)
    yr_range = f"{years[0]}-{years[-1]}"
    print(f"{label:<10}{yr_range:<14}{len(years):>6}{rf_mape:>11.2f}%{dt_mape:>11.2f}%")

# Detailed test-year breakdown (the headline forecast comparison)
print(f"\nDetailed test-year breakdown (RF aggregate forecast):")
print(f"  {'Year':<6}{'Sum actual':>12}{'Sum RF pred':>14}{'|err|':>10}{'|err|/actual':>16}")
print("  " + "-" * 56)
years_te, actual_te, rf_te, dt_te = aggregate_split('te')
for y, a, r in zip(years_te, actual_te, rf_te):
    err = abs(a - r)
    rel = err / a if a != 0 else np.nan
    print(f"  {int(y):<6}{int(a):>12}{r:>14.2f}{err:>10.2f}{rel*100:>15.2f}%")

print(f"\n{'='*80}")
print("DONE")
print(f"{'='*80}")


STEP 10 — Aggregate cluster forecasts → aggregate train/val/test MAPE

Split     Years           #yrs     RF MAPE     DT MAPE
--------------------------------------------------------
Train     1977-2014         38      15.83%      45.50%
Val       2015-2019          5      17.42%      10.47%
Test      2020-2024          5       8.63%       8.62%

Detailed test-year breakdown (RF aggregate forecast):
  Year    Sum actual   Sum RF pred     |err|    |err|/actual
  --------------------------------------------------------
  2020          1391       1323.11     67.89           4.88%
  2021          1404       1323.37     80.63           5.74%
  2022          1418       1358.06     59.94           4.23%
  2023          1208       1362.80    154.80          12.81%
  2024          1549       1309.46    239.54          15.46%

DONE
